# Практика 14. Параметричні критерії: t-критерій для однієї та двох вибірок, порівняння дисперсій

**Студент:** Гузь Юрій  
**Група:** ІТ-41  
**Варіант:** 3, Одеса, Південь  
**Довідкова норма вологості:** 73%

Мета: застосувати одновибірковий, незалежний двовибірковий і парний t-критерії, а перед незалежним t-тестом перевірити рівність дисперсій критерієм Лівена.

## Дані варіанта

Нижче наведено 12 середньомісячних значень відносної вологості Одеси о 8:00. Їх використовуємо для всіх завдань.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

alpha = 0.05
humidity_by_city = {
    "Київ": {
        "region": "Центр",
        "values": [84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85],
    },
    "Львів": {
        "region": "Захід",
        "values": [85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86],
    },
    "Одеса": {
        "region": "Південь",
        "values": [82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83],
    },
    "Харків": {
        "region": "Схід",
        "values": [83, 80, 75, 68, 62, 60, 58, 60, 66, 73, 79, 84],
    },
    "Дніпро": {
        "region": "Схід",
        "values": [82, 79, 74, 67, 61, 59, 57, 59, 65, 72, 78, 83],
    },
    "Чернігів": {
        "region": "Центр",
        "values": [85, 83, 79, 71, 66, 63, 61, 63, 69, 76, 82, 86],
    },
    "Івано-Франківськ": {
        "region": "Захід",
        "values": [86, 84, 81, 75, 70, 67, 65, 67, 72, 78, 83, 87],
    },
    "Полтава": {
        "region": "Центр",
        "values": [84, 81, 76, 69, 63, 61, 59, 61, 67, 74, 80, 85],
    },
    "Суми": {
        "region": "Схід",
        "values": [85, 82, 77, 70, 64, 62, 60, 62, 68, 75, 81, 86],
    },
    "Ужгород": {
        "region": "Захід",
        "values": [84, 82, 79, 73, 68, 66, 64, 66, 71, 77, 82, 85],
    },
}

city = "Одеса"
norm = 73
morning = np.array(humidity_by_city[city]["values"], dtype=float)
print(f"Місто: {city}; регіон: {humidity_by_city[city]['region']}; норма: {norm}%")
print("Вологість о 8:00 за місяцями (%):", morning)

## Завдання 1. Одновибірковий t-критерій проти норми

Перевіримо двобічну гіпотезу: **H₀:** середня вологість Одеси дорівнює довідковій нормі 73%; **H₁:** середня вологість відрізняється від 73%. Двобічний тест обрано, бо цікавить будь-яка відмінність від норми.

In [ ]:
mean_humidity = morning.mean()
std_humidity = morning.std(ddof=1)
n = len(morning)
t_one, p_one = stats.ttest_1samp(morning, popmean=norm)
reject_one = p_one < alpha

print(f"x̄ = {mean_humidity:.3f}%, s = {std_humidity:.3f}%, n = {n}")
print(f"t({n - 1}) = {t_one:.4f}, p-value = {p_one:.6f}")
print("Рішення:", "відхиляємо H₀" if reject_one else "не відхиляємо H₀")
if reject_one:
    print(f"За α={alpha:.2f} є статистично значущі підстави вважати середню місячну вологість Одеси відмінною від норми {norm}%.")
else:
    print(f"За α={alpha:.2f} недостатньо статистичних підстав вважати середню місячну вологість Одеси відмінною від норми {norm}%; це не доводить точну рівність.")

## Завдання 2. Ручна перевірка t-статистики

Перевіримо формулу `t = (x̄ - μ₀) / (s / √n)` за тими самими даними.

In [ ]:
se = std_humidity / np.sqrt(n)
t_manual = (mean_humidity - norm) / se

print(f"Стандартна похибка: {se:.6f}")
print(f"Ручна t-статистика: {t_manual:.6f}")
print(f"ttest_1samp:         {t_one:.6f}")
print("Збігаються з точністю до округлення:", np.isclose(t_manual, t_one))

## Завдання 3. Незалежні вибірки: Одеса проти Києва

Для порівняння обрано Київ (регіон Центр), який належить до іншого регіону, ніж Одеса (Південь). Спочатку критерієм Лівена перевіряємо рівність дисперсій; лише після цього обираємо варіант незалежного t-тесту.

In [ ]:
city_3 = "Київ"
sample_3 = np.array(humidity_by_city[city_3]["values"], dtype=float)
levene_3, p_levene_3 = stats.levene(morning, sample_3)
equal_var_3 = p_levene_3 >= alpha
t_ind_3, p_ind_3 = stats.ttest_ind(morning, sample_3, equal_var=equal_var_3)

print(f"Порівнювані міста: {city} ({humidity_by_city[city]['region']}) і {city_3} ({humidity_by_city[city_3]['region']})")
print(f"Вибіркові стандартні відхилення: Одеса={morning.std(ddof=1):.3f}, {city_3}={sample_3.std(ddof=1):.3f}")
print(f"Levene: statistic={levene_3:.4f}, p={p_levene_3:.6f}")
print("Висновок про дисперсії:", "немає підстав вважати дисперсії різними" if equal_var_3 else "є підстави вважати дисперсії різними")
print("Варіант t-тесту:", "стандартний (equal_var=True)" if equal_var_3 else "Welch (equal_var=False)")
print(f"t-тест: t={t_ind_3:.4f}, p={p_ind_3:.6f}")
print("Висновок:", "середні вологості статистично значуще відрізняються" if p_ind_3 < alpha else "немає достатніх підстав стверджувати, що середні вологості відрізняються")

## Завдання 4. Інше місто з протилежним результатом Лівена

Автоматично перевіримо міста інших регіонів (не Центр і не Південь) і візьмемо перше, для якого рішення критерію Лівена при α=0.05 протилежне рішенню для Завдання 3. Це дає змогу обрати місто на основі фактичного результату `levene()`, а не вгадувати за стандартними відхиленнями. Якщо серед кандидатів немає такого міста, код повідомить про це явно.

In [ ]:
opposite_candidate = None
levene_4 = p_levene_4 = None

for candidate, details in humidity_by_city.items():
    if candidate in {city, city_3} or details["region"] in {humidity_by_city[city]["region"], humidity_by_city[city_3]["region"]}:
        continue
    candidate_sample = np.array(details["values"], dtype=float)
    candidate_stat, candidate_p = stats.levene(morning, candidate_sample)
    candidate_equal_var = candidate_p >= alpha
    if candidate_equal_var != equal_var_3:
        opposite_candidate = candidate
        levene_4, p_levene_4 = candidate_stat, candidate_p
        break

if opposite_candidate is None:
    raise RuntimeError("Не знайдено місто іншого регіону з протилежним висновком Лівена; перевірте таблицю даних або рівень α.")

sample_4 = np.array(humidity_by_city[opposite_candidate]["values"], dtype=float)
equal_var_4 = p_levene_4 >= alpha
t_ind_4, p_ind_4 = stats.ttest_ind(morning, sample_4, equal_var=equal_var_4)

print(f"Обране місто: {opposite_candidate} ({humidity_by_city[opposite_candidate]['region']})")
print(f"Вибіркові стандартні відхилення: Одеса={morning.std(ddof=1):.3f}, {opposite_candidate}={sample_4.std(ddof=1):.3f}")
print(f"Levene: statistic={levene_4:.4f}, p={p_levene_4:.6f}")
print("Висновок про дисперсії:", "немає підстав вважати дисперсії різними" if equal_var_4 else "є підстави вважати дисперсії різними")
print("Варіант t-тесту:", "стандартний (equal_var=True)" if equal_var_4 else "Welch (equal_var=False)")
print(f"t-тест: t={t_ind_4:.4f}, p={p_ind_4:.6f}")
print("Висновок:", "середні вологості статистично значуще відрізняються" if p_ind_4 < alpha else "немає достатніх підстав стверджувати, що середні вологості відрізняються")

У Завданні 4 висновок Лівена протилежний до Завдання 3, тому змінюється і вибір `equal_var`: за `p ≥ 0.05` застосовуємо стандартний двовибірковий t-тест, а за `p < 0.05` — Welch-тест. Порівняння `std()` допомагає побачити різницю розкидів, але саме рішення приймаємо за формальним критерієм Лівена.

## Завдання 5. Парний t-критерій: ранок проти вечора

Для кожного місяця вечірня вологість отримується відніманням заданої добової амплітуди від ранкового значення. Це парні спостереження за той самий місяць, тому тестуємо середнє внутрішньомісячних різниць за допомогою `ttest_rel()`.

In [ ]:
daily_amplitude = np.array([3, 3, 4, 5, 6, 7, 7, 7, 6, 5, 4, 3], dtype=float)
evening = morning - daily_amplitude
t_rel, p_rel = stats.ttest_rel(morning, evening)
reject_rel = p_rel < alpha

print("Ранкові значення (%):", morning)
print("Вечірні значення (%):", evening)
print("Різниця ранок − вечір (%):", morning - evening)
print(f"Середня парна різниця: {(morning - evening).mean():.3f} п.п.")
print(f"Парний t-тест: t={t_rel:.4f}, p={p_rel:.6f}")
print("Висновок:", "добова різниця статистично значуща" if reject_rel else "немає достатніх підстав стверджувати, що добова різниця відмінна від нуля")

Парний критерій доречний, бо кожне ранкове значення зіставляється з вечірнім значенням того самого місяця. Обчислення різниці для кожної пари контролює спільні для місяця умови (наприклад, сезонний рівень вологості) і зосереджується на добовій зміні; незалежний t-тест помилково ігнорував би цей зв'язок між вимірюваннями.

## Контрольні питання

1. **Чому для середнього використовують t-, а не z-розподіл?** Генеральне стандартне відхилення σ невідоме й оцінюється за вибірковим `s`. t-розподіл враховує додаткову невизначеність цієї оцінки; для одновибіркового тесту він має `n-1` ступенів вільності.

2. **Чому важливий вибір `equal_var`?** Стандартний `ttest_ind` об'єднує оцінки дисперсій і розраховує стандартну похибку за припущення рівних дисперсій. Якщо це припущення не виконується, стандартна похибка й ступені вільності можуть бути оцінені неправильно, що здатне спотворити p-value. Welch-варіант не вимагає рівності дисперсій.

3. **Чим парний критерій може бути потужнішим?** Парний тест аналізує різниці в межах пов'язаних пар, усуваючи спільну для пари мінливість. Якщо ця спільна мінливість значна, розкид різниць і стандартна похибка середньої різниці зменшуються порівняно з аналізом тих самих даних як незалежних.

4. **Чи збігається Лівен із порівнянням `std()`?** `std()` показує відносну величину розкиду в кожній вибірці, тоді як критерій Лівена оцінює статистичні докази відмінності дисперсій. Тому більша різниця стандартних відхилень може узгоджуватися з малим p-value Лівена, але візуального порівняння самого по собі недостатньо для висновку.

## Підсумок

Одновибірковий t-критерій порівняв середню вологість Одеси з довідковою нормою, ручна формула підтвердила його t-статистику, а незалежні вибірки порівнювалися після попередньої перевірки дисперсій критерієм Лівена. Для пов'язаних ранкових і вечірніх місячних вимірювань застосовано парний t-критерій.